In [1]:
# Celda 1 — imports
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

sys.path.append(str(Path('..') / 'src'))
from config import CH, MY_ENGINE, BRONZE_DB

In [ ]:
# Celda 2 — función de carga con query explícita
def cargar_a_bronze(nombre_tabla, query):
    print(f"Leyendo {nombre_tabla} desde MySQL...")
    df = pd.read_sql(query, MY_ENGINE)

    df['_ingested_at'] = datetime.now()
    df['_source']      = 'mysql'
    df = df.where(pd.notnull(df), None)
    df = df.where(df.notna(), None)

    
    if nombre_tabla == 'rental':
        df['return_date'] = df['return_date'].where(df['return_date'].notna(), None)
        df['return_date'] = df['return_date'].astype(object).where(df['return_date'].notna(), None)
    
    # Convertir los valores a lista de listas con tipos nativos
    data = []
    for row in df.values.tolist():
        clean_row = []
        for val in row:
            if isinstance(val, float) and np.isnan(val):
                clean_row.append(None)
            elif isinstance(val, (np.integer,)):
                clean_row.append(int(val))
            elif isinstance(val, (np.floating,)):
                clean_row.append(float(val))
            else:
                clean_row.append(val)
        data.append(clean_row)

    CH.command(f"TRUNCATE TABLE {BRONZE_DB}.raw_{nombre_tabla}")
    CH.insert(f"{BRONZE_DB}.raw_{nombre_tabla}",
            data,
            column_names=df.columns.tolist())
    print(f"✓ {len(df):,} filas cargadas en bronze.raw_{nombre_tabla}")

In [3]:
tablas = {
    'actor':         "SELECT actor_id, first_name, last_name, last_update FROM actor",
    'address': "SELECT address_id, address, address2, district, city_id, postal_code, phone, ST_AsText(location) AS location, last_update FROM address",
    'category':      "SELECT category_id, name, last_update FROM category",
    'city':          "SELECT city_id, city, country_id, last_update FROM city",
    'country':       "SELECT country_id, country, last_update FROM country",
    'customer':      "SELECT customer_id, store_id, first_name, last_name, email, address_id, active, create_date, last_update FROM customer",
    'film':          "SELECT film_id, title, description, release_year, language_id, original_language_id, rental_duration, rental_rate, length, replacement_cost, rating, special_features, last_update FROM film",
    'film_actor':    "SELECT actor_id, film_id, last_update FROM film_actor",
    'film_category': "SELECT film_id, category_id, last_update FROM film_category",
    'film_text':     "SELECT film_id, title, description FROM film_text",
    'inventory':     "SELECT inventory_id, film_id, store_id, last_update FROM inventory",
    'language':      "SELECT language_id, name, last_update FROM language",
    'payment':       "SELECT payment_id, customer_id, staff_id, rental_id, amount, payment_date, last_update FROM payment",
    'rental':        "SELECT rental_id, rental_date, inventory_id, customer_id, return_date, staff_id, last_update FROM rental",
    'staff':         "SELECT staff_id, first_name, last_name, address_id, picture, email, store_id, active, username, password, last_update FROM staff",
    'store':         "SELECT store_id, manager_staff_id, address_id, last_update FROM store",
}

In [15]:
for tabla, query in tablas.items():
    cargar_a_bronze(tabla, query)

Leyendo actor desde MySQL...
✓ 200 filas cargadas en bronze.raw_actor
Leyendo address desde MySQL...
✓ 603 filas cargadas en bronze.raw_address
Leyendo category desde MySQL...
✓ 16 filas cargadas en bronze.raw_category
Leyendo city desde MySQL...
✓ 600 filas cargadas en bronze.raw_city
Leyendo country desde MySQL...
✓ 109 filas cargadas en bronze.raw_country
Leyendo customer desde MySQL...
✓ 599 filas cargadas en bronze.raw_customer
Leyendo film desde MySQL...
✓ 1,000 filas cargadas en bronze.raw_film
Leyendo film_actor desde MySQL...
✓ 5,462 filas cargadas en bronze.raw_film_actor
Leyendo film_category desde MySQL...
✓ 1,000 filas cargadas en bronze.raw_film_category
Leyendo film_text desde MySQL...
✓ 0 filas cargadas en bronze.raw_film_text
Leyendo inventory desde MySQL...
✓ 4,581 filas cargadas en bronze.raw_inventory
Leyendo language desde MySQL...
✓ 6 filas cargadas en bronze.raw_language
Leyendo payment desde MySQL...
✓ 14,881 filas cargadas en bronze.raw_payment
Leyendo rental d